In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-ibm-catalog scikit-learn
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Singularity Machine Learning - Klasifikace: Qiskit Function od Multiverse Computing
> **Note:** * Qiskit Functions jsou experimentální funkce dostupné pouze uživatelům plánů IBM Quantum&reg; Premium Plan, Flex Plan a On-Prem (přes IBM Quantum Platform API). Jsou ve fázi preview a mohou se měnit.

## Přehled
Funkce „Singularity Machine Learning - Klasifikace" ti umožňuje řešit skutečné problémy strojového učení na kvantovém hardwaru, aniž bys potřeboval/a znalosti kvantové fyziky. Tato aplikační funkce, založená na metodách ensemble, je hybridní klasifikátor. Využívá klasické metody jako boosting, bagging a stacking pro počáteční trénování ensemblu. Následně jsou použity kvantové algoritmy, jako je variační kvantový eigensolver (VQE) a kvantový aproximační optimalizační algoritmus (QAOA), aby se zvýšila rozmanitost natrénovaného ensemblu, jeho schopnost generalizace a celková složitost.

Na rozdíl od jiných řešení kvantového strojového učení je tato funkce schopna zpracovávat rozsáhlé datové sady s miliony příkladů a příznaků, aniž by byla omezena počtem qubitů v cílovém QPU. Počet qubitů určuje pouze velikost ensemblu, který lze natrénovat. Je také velmi flexibilní a lze ji použít k řešení klasifikačních problémů v mnoha oblastech, včetně financí, zdravotnictví a kybernetické bezpečnosti.
Konzistentně dosahuje vysoké přesnosti u klasicky náročných problémů zahrnujících vícerozměrné, zašuměné a nevyvážené datové sady.
![Jak to funguje](../docs/images/guides/multiverse-computing-singularity/how_it_works.avif)
Je určena pro:
1. Inženýry a datové vědce ve firmách, kteří chtějí rozšířit svou technologickou nabídku integrací kvantového strojového učení do svých produktů a služeb,
2. Výzkumníky v kvantových výzkumných laboratořích, kteří zkoumají aplikace kvantového strojového učení a chtějí využít kvantové výpočty pro klasifikační úlohy, a
3. Studenty a učitele na vzdělávacích institucích v kurzech strojového učení, kteří chtějí demonstrovat výhody kvantových výpočtů.

Následující příklad ukazuje různé funkce, včetně `create`, `list`, `fit` a `predict`, a demonstruje jejich použití na syntetickém problému dvou proložených půlkružnic – notoricky náročném problému kvůli jeho nelineární rozhodovací hranici.


## Popis funkce
Tato Qiskit Function umožňuje uživatelům řešit problémy binární klasifikace pomocí kvantově vylepšeného ensemble klasifikátoru Singularity. Za scénou využívá hybridní přístup ke klasickému trénování ensemblu klasifikátorů na označené datové sadě a poté ho optimalizuje pro maximální rozmanitost a generalizaci pomocí Kvantového aproximačního optimalizačního algoritmu (QAOA) na IBM&reg; QPU. Prostřednictvím uživatelsky přívětivého rozhraní mohou uživatelé konfigurovat klasifikátor podle svých požadavků, natrénovat ho na datové sadě dle vlastního výběru a použít ho k predikci na dříve neviděné datové sadě.

Postup řešení obecného klasifikačního problému:
1. Předzpracuj datovou sadu a rozděl ji na trénovací a testovací sady. Volitelně můžeš trénovací sadu dále rozdělit na trénovací a validační sadu. Toho lze dosáhnout pomocí [scikit-learn](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.train_test_split.html).
2. Pokud je trénovací sada nevyvážená, můžeš ji převzorkovat, aby byly třídy vyvážené, pomocí [imbalanced-learn](https://imbalanced-learn.org/stable/introduction.html#problem-statement-regarding-imbalanced-data-sets).
3. Nahraj trénovací, validační a testovací sady samostatně do úložiště funkce pomocí metody `file_upload` katalogu a pokaždé jí předej příslušnou cestu.
4. Inicializuj kvantový klasifikátor pomocí akce `create` funkce, která přijímá hyperparametry, jako je počet a typy modelů, regularizace (hodnota lambda) a možnosti optimalizace včetně počtu vrstev, typu klasického optimalizátoru, kvantového Backend a tak dále.
5. Natrénuj kvantový klasifikátor na trénovací sadě pomocí akce `fit` funkce, které předáš označenou trénovací sadu a případně validační sadu.
6. Proveď predikci na dříve neviděné testovací sadě pomocí akce `predict` funkce.

## Přístup založený na akcích
Funkce využívá přístup založený na akcích. Můžeš si ji představit jako virtuální prostředí, kde pomocí akcí provádíš úkoly nebo měníš její stav. V současnosti nabízí tyto akce: [list](#1-list), [create](#2-create), [delete](#3-delete), [fit](#4-fit), [predict](#5-predict), [fit_predict](#6-fit-predict) a [create_fit_predict](#7-create-fit-predict). Následující příklad demonstruje akci `create_fit_predict`.

In [ ]:
# Import QiskitFunctionsCatalog to load the
# "Singularity Machine Learning - Classification" function by Multiverse Computing
from qiskit_ibm_catalog import QiskitFunctionsCatalog

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# authentication
# If you have not previously saved your credentials, follow instructions at
# /docs/guides/functions
# to authenticate with your API key.
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load "Singularity Machine Learning - Classification" function by Multiverse Computing
singularity = catalog.load("multiverse/singularity")

# generate the synthetic dataset
X, y = make_moons(n_samples=1000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

job = singularity.run(
    action="create_fit_predict",
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    options={"save": False},
)

# get job status and result
status = job.status()
result = job.result()

print("Job status: ", status)
print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results): ", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results): ",
    result["data"]["probabilities"][:5],
)
print("Usage metadata: ", result["metadata"]["resource_usage"])

Job status:  QUEUED
Action result status:  ok
Action result message:  Classifier created, fitted, and predicted.
Predictions (first five results):  [1, 0, 0, 1, 0]
Probabilities (first five results):  [[0.16849563539001172, 0.8315043646099888], [0.8726393386620336, 0.12736066133796647], [0.795344837290717, 0.20465516270928288], [0.36822585748882725, 0.6317741425111725], [0.6656662698604361, 0.3343337301395641]]
Usage metadata:  {'RUNNING: MAPPING': {'CPU_TIME': 7.945035696029663}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 82.41029238700867}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 77.3459484577179}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 71.27004957199097}}


### 1. List

The `list` action retrieves all stored classifiers in `*.pkl.tar` format from the shared data directory. You can also access the contents of this directory by using the `catalog.files()` method. In general, the list action searches for files with the `*.pkl.tar` extension in the shared data directory and returns them in a list format.

#### Inputs

|     Name    |    Type     | Description |   Required  |
|-------------|-------------|-------------|-------------|
| `action` | `str` | The name of the action from among `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` and `delete`. | Yes |

#### Usage

In [ ]:
job = singularity.run(action="list")

### 1. List

Akce `list` načte všechny uložené klasifikátory ve formátu `*.pkl.tar` ze sdíleného datového adresáře. Obsah tohoto adresáře můžeš také zobrazit pomocí metody `catalog.files()`. Obecně platí, že akce list vyhledá soubory s příponou `*.pkl.tar` ve sdíleném datovém adresáři a vrátí je ve formátu seznamu.

#### Vstupy

|     Název    |    Typ     | Popis |   Povinný  |
|-------------|-------------|-------------|-------------|
| `action` | `str` | Název akce z možností `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` a `delete`. | Ano |

#### Použití

In [ ]:
job = singularity.run(
    action="create",
    name="classifier_name",  # specify your custom name for the classifier here
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
)

### 2. Create
Akce `create` vytvoří klasifikátor zadaného typu `quantum_classifier` pomocí poskytnutých parametrů a uloží ho ve sdíleném datovém adresáři.

> **Note:** Funkce aktuálně podporuje pouze `QuantumEnhancedEnsembleClassifier`.
#### Vstupy
|     Název    |    Typ     | Popis | Povinný  | Výchozí |
|-------------|-------------|-------------|-----------|---------|
| `action` | `str` | Název akce z možností `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` a `delete`. | Ano | - |
| `name` | `str` | Název kvantového klasifikátoru, např. `spam_classifier`. | Ano | - |
| `instance` | `str` | Instance IBM. | Ano | - |
| `backend_name` | `str` | Výpočetní prostředek IBM. Výchozí hodnota je `None`, což znamená, že bude použit Backend s nejmenším počtem čekajících úloh. | Ne | `None` |
| `quantum_classifier` | `str` | Typ kvantového klasifikátoru, tj. `QuantumEnhancedEnsembleClassifier`. | Ne | `QuantumEnhancedEnsembleClassifier` |
| `num_learners` | `integer` | Počet learnerů v ansámblu. | Ne | `10` |
| `learners_types` | `list` | Typy learnerů. Mezi podporované typy patří: `DecisionTreeClassifier`, `GaussianNB`, `KNeighborsClassifier`, `MLPClassifier` a `LogisticRegression`. Další podrobnosti ke každému z nich najdeš v [dokumentaci scikit-learn](https://scikit-learn.org/stable/supervised_learning.html). | Ne | `[DecisionTreeClassifier]` |
| `learners_proportions` | `list` | Proporce každého typu learneru v ansámblu. | Ne | `[1.0]` |
| `learners_options` | `list` | Možnosti pro každý typ learneru v ansámblu. Úplný seznam možností odpovídajících zvolenému typu learneru najdeš v [dokumentaci scikit-learn](https://scikit-learn.org/stable/supervised_learning.html). | Ne | `[{"max_depth": 3, "splitter": "random", "class_weight": None}]` |
| `regularization_type` | `str` nebo `list` | Typ/y regularizace, která má být použita: `onsite` nebo `alpha`. `onsite` řídí člen onsite – vyšší hodnoty vedou k řidším ansámblům. `alpha` řídí kompromis mezi členy interakce a onsite – nižší hodnoty vedou k řidším ansámblům. Pokud je zadán seznam, budou modely trénovány pro každý typ a bude vybrán ten s nejlepším výkonem. | Ne | `onsite` |
| `regularization` | `str` nebo `float` nebo `list` | Hodnota regularizace. Ohraničena mezi `0` a `+inf`, pokud je regularization_type `onsite`. Ohraničena mezi `0` a `1`, pokud je regularization_type `alpha`. Pokud je nastavena na `auto`, použije se automatická regularizace – optimální regularizační parametr je nalezen binárním prohledáváním s požadovaným poměrem vybraných klasifikátorů k celkovému počtu klasifikátorů (`regularization_desired_ratio`) a horní mezí regularizačního parametru (`regularization_upper_bound`). Pokud je zadán seznam, budou modely trénovány pro každou hodnotu a bude vybrán ten s nejlepším výkonem. | Ne | `0.01` |
| `regularization_desired_ratio` | `float` nebo `list` | Požadovaný poměr/y vybraných klasifikátorů k celkovému počtu klasifikátorů pro automatickou regularizaci. Pokud je zadán seznam, budou modely trénovány pro každý poměr a bude vybrán ten s nejlepším výkonem. | Ne | `0.75` |
| `regularization_upper_bound` | `float` nebo `list` | Horní mez/e regularizačního parametru při použití automatické regularizace. Pokud je zadán seznam, budou modely trénovány pro každou horní mez a bude vybrán ten s nejlepším výkonem. | Ne | `200` |
| `weight_update_method` | `str` | Metoda aktualizace vah vzorků z možností `logarithmic` a `quadratic`. | Ne | `logarithmic` |
| `sample_scaling` | `boolean` | Zda má být použito škálování vzorků. | Ne | `False` |
| `prediction_scaling` | `float` | Škálovací faktor pro predikce. | Ne | `None` |
| `optimizer_options` | `dictionary` | Možnosti optimalizátoru QAOA. Seznam dostupných možností je uveden dále v této dokumentaci. | Ne | ... |
| `voting` | `str` | Použij hlasování nadpoloviční většinou (`hard`) nebo průměr pravděpodobností (`soft`) pro agregaci predikcí/pravděpodobností learnerů. | Ne | `hard` |
| `prob_threshold` | `float` | Optimální práh pravděpodobnosti. | Ne | `0.5` |
| `random_state` | `integer` | Řízení náhodnosti pro opakovatelnost. | Ne | `None` |


- Dále jsou uvedeny možnosti `optimizer_options`:

|     Název    |    Typ     | Popis | Povinný  | Výchozí |
|-------------|-------------|-------------|-----------|---------|
| `num_solutions` | `integer` | Počet řešení | Ne | `1024` |
| `reps` | `integer` | Počet opakování | Ne | `4` |
| `sparsify` | `float` | Práh řídnutí | Ne | `0.001` |
| `theta` | `float` | Počáteční hodnota theta, variačního parametru QAOA | Ne | `None` |
| `simulator` | `boolean` | Zda použít simulátor nebo QPU | Ne | `False` |
| `classical_optimizer` | `str` | Název klasického optimalizátoru pro QAOA. Lze použít všechny řešiče nabízené SciPy, uvedené [zde](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html#scipy.optimize.minimize). Podle toho bude třeba nastavit `classical_optimizer_options`. | Ne | `COBYLA` |
| `classical_optimizer_options` | `dictionary` | Možnosti klasického optimalizátoru. Úplný seznam dostupných možností najdeš v [dokumentaci SciPy](https://docs.scipy.org/doc/scipy/reference/). | Ne | `{"maxiter": 60}` |
| `optimization_level` | `integer` | Hloubka Circuit QAOA | Ne | `3` |
| `num_transpiler_runs` | `integer` | Počet spuštění Transpileru | Ne | `30` |
| `pass_manager_options` | `dictionary` | Možnosti pro generování přednastavené správy průchodů | Ne | `{"approximation_degree": 1.0}` |
| `estimator_options` | `dictionary` | Možnosti Estimatoru. Úplný seznam dostupných možností najdeš v [dokumentaci Qiskit Runtime Client](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options). | Ne | `None` |
| `sampler_options` | `dictionary` | Možnosti Sampleru. Úplný seznam dostupných možností najdeš v [dokumentaci Qiskit Runtime Client](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options). | Ne | `None` |

- Výchozí `estimator_options` jsou:

|     Název    |    Typ     | Hodnota  |
|-------------|-------------|-------------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `2` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |
| `resilience_options` | `dictionary` | `{"zne_mitigation": False, "zne": {"amplifier": "pea", "noise_factors": [1.0, 1.3, 1.6], "extrapolator": ["linear", "polynomial_degree_2", "exponential"],}}` |

- Výchozí `sampler_options` jsou:

|     Název    |    Typ     | Hodnota |
|-------------|-------------|-------------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `1` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |

#### Použití

In [ ]:
job = singularity.run(
    action="delete",
    name="classifier_name",  # specify the name of the classifier to delete here
)

#### Validace
- `name`:
    - Název musí být jedinečný, řetězec o délce nejvýše 64 znaků.
    - Může obsahovat pouze alfanumerické znaky a podtržítka.
    - Musí začínat písmenem a nesmí končit podtržítkem.
    - Ve sdíleném datovém adresáři nesmí existovat žádný klasifikátor se stejným názvem.

### 3. Delete
Akce `delete` odstraní klasifikátor ze sdíleného datového adresáře.

#### Vstupy
|     Název    |    Typ     | Popis |   Povinný  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | Název akce. Musí být `delete`. | Ano |
| `name`      | `str`    | Název klasifikátoru, který má být smazán. | Ano |

#### Použití

In [ ]:
job = singularity.run(
    action="fit",
    name="classifier_name",  # specify the name of the classifier to train here
    X=X_train,  # or "X_train.npy" if you uploaded it in the shared data directory
    y=y_train,  # or "y_train.npy" if you uploaded it in the shared data directory
    fit_params={},  # define the fit parameters here
)

#### Validace
- `name`:
    - Název musí být jedinečný, řetězec o délce nejvýše 64 znaků.
    - Může obsahovat pouze alfanumerické znaky a podtržítka.
    - Musí začínat písmenem a nesmí končit podtržítkem.
    - Klasifikátor se stejným názvem musí již existovat ve sdíleném datovém adresáři.
### 4. Fit
Akce `fit` trénuje klasifikátor pomocí poskytnutých trénovacích dat.

#### Vstupy
|     Název   |    Typ      | Popis |   Povinný   |
|-------------|-------------|-------|-------------|
| `action`    | `str`    | Název akce. Musí být `fit`. | Ano |
| `name`      | `str`    | Název klasifikátoru, který se má trénovat. | Ano |
| `X`         | `array` nebo `list` nebo `str` | Trénovací data. Může jít o NumPy pole, seznam nebo řetězec odkazující na název souboru ve sdíleném datovém adresáři. | Ano |
| `y`         | `array` nebo `list` nebo `str` | Cílové hodnoty trénovacích dat. Může jít o NumPy pole, seznam nebo řetězec odkazující na název souboru ve sdíleném datovém adresáři. | Ano |
| `fit_params`| `dictionary`| Další parametry předané metodě `fit` klasifikátoru. | Ne |

##### fit_params
|     Název   |    Typ      | Popis |   Povinný   |   Výchozí   |
|-------------|-------------|-------|-------------|-------------|
| `validation_data` | `tuple` | Validační data a štítky. | Ne | `None` |
| `pos_label` | `integer` nebo `str` | Štítek třídy, který se mapuje na hodnotu 1. | Ne | `None` |
| `optimization_data` | `str` | Datová sada použitá pro optimalizaci ansámblu. Může být jedna z: `train`, `validation`, `both`. | Ne | `train` |

#### Použití

In [ ]:
job = singularity.run(
    action="predict",
    name="classifier_name",  # specify the name of the classifier to use here
    X=X_test,  # or "X_test.npy" if you uploaded it to the shared data directory
    options={
        "out": "output.json",
    },
)

#### Validace
- `name`:
    - Název musí být jedinečný, řetězec o délce maximálně 64 znaků.
    - Může obsahovat pouze alfanumerické znaky a podtržítka.
    - Musí začínat písmenem a nesmí končit podtržítkem.
    - Klasifikátor se stejným názvem musí již existovat ve sdíleném datovém adresáři.
### 5. Predict
Akce `predict` slouží k získání tvrdých a měkkých predikcí (pravděpodobností).

#### Vstupy
|     Název   |    Typ      | Popis |   Povinný   |
|-------------|-------------|-------|-------------|
| `action`    | `str`    | Název akce. Musí být `predict`. | Ano |
| `name`      | `str`    | Název klasifikátoru, který se má použít. | Ano |
| `X`         | `array` nebo `list` nebo `str` | Testovací data. Může jít o NumPy pole, seznam nebo řetězec odkazující na název souboru ve sdíleném datovém adresáři. | Ano |
| `options["out"]` | `str` | Název výstupního JSON souboru pro uložení predikcí ve sdíleném datovém adresáři. Pokud není zadán, predikce jsou vráceny ve výsledku úlohy. | Ne |

#### Použití

In [ ]:
job = singularity.run(
    action="fit_predict",
    name="classifier_name",  # specify the name of the classifier to use here
    X_train=X_train,  # or "X_train.npy" if you uploaded it in the shared data directory
    y_train=y_train,  # or "y_train.npy" if you uploaded it in the shared data directory
    X_test=X_test,  # or "X_test.npy" if you uploaded it in the shared data directory
    fit_params={},  # define the fit parameters here
    options={
        "out": "output.json",
    },
)

#### Validace
- `name`:
    - Název musí být jedinečný, řetězec o délce maximálně 64 znaků.
    - Může obsahovat pouze alfanumerické znaky a podtržítka.
    - Musí začínat písmenem a nesmí končit podtržítkem.
    - Klasifikátor se stejným názvem musí již existovat ve sdíleném datovém adresáři.
- `options["out"]`:
    - Název souboru musí být jedinečný, řetězec o délce maximálně 64 znaků.
    - Může obsahovat pouze alfanumerické znaky a podtržítka.
    - Musí začínat písmenem a nesmí končit podtržítkem.
    - Musí mít příponu `.json`.
### 6. Fit-predict
Akce `fit_predict` trénuje klasifikátor pomocí trénovacích dat a následně ho použije k získání tvrdých a měkkých predikcí (pravděpodobností).

#### Vstupy
|     Název   |    Typ      | Popis |   Povinný   |
|-------------|-------------|-------|-------------|
| `action`    | `str`    | Název akce. Musí být `fit_predict`. | Ano |
| `name`      | `str`    | Název klasifikátoru, který se má použít. | Ano |
| `X_train`   | `array` nebo `list` nebo `str` | Trénovací data. Může jít o NumPy pole, seznam nebo řetězec odkazující na název souboru ve sdíleném datovém adresáři. | Ano |
| `y_train`   | `array` nebo `list` nebo `str` | Cílové hodnoty trénovacích dat. Může jít o NumPy pole, seznam nebo řetězec odkazující na název souboru ve sdíleném datovém adresáři. | Ano |
| `X_test`    | `array` nebo `list` nebo `str` | Testovací data. Může jít o NumPy pole, seznam nebo řetězec odkazující na název souboru ve sdíleném datovém adresáři. | Ano |
| `fit_params`| `dictionary`| Další parametry předané metodě `fit` klasifikátoru. | Ne |
| `options["out"]` | `str` | Název výstupního JSON souboru pro uložení predikcí ve sdíleném datovém adresáři. Pokud není zadán, predikce jsou vráceny ve výsledku úlohy. | Ne |

#### Použití

In [ ]:
job = singularity.run(
    action="create_fit_predict",
    name="classifier_name",  # specify your custom name for the classifier here
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,  # or "X_train.npy" if you uploaded it in the shared data directory
    y_train=y_train,  # or "y_train.npy" if you uploaded it in the shared data directory
    X_test=X_test,  # or "X_test.npy" if you uploaded it in the shared data directory
    fit_params={},  # define the fit parameters here
    options={
        "save": True,
        "out": "output.json",
    },
)

#### Validace
- `name`:
    - Název musí být jedinečný, řetězec o délce maximálně 64 znaků.
    - Může obsahovat pouze alfanumerické znaky a podtržítka.
    - Musí začínat písmenem a nesmí končit podtržítkem.
    - Klasifikátor se stejným názvem musí již existovat ve sdíleném datovém adresáři.

- `options["out"]`:
    - Název souboru musí být jedinečný, řetězec o délce maximálně 64 znaků.
    - Může obsahovat pouze alfanumerické znaky a podtržítka.
    - Musí začínat písmenem a nesmí končit podtržítkem.
    - Musí mít příponu `.json`.
### 7. Create-fit-predict
Akce `create_fit_predict` vytvoří klasifikátor, natrénuje ho pomocí poskytnutých trénovacích dat a poté ho použije k získání pevných a měkkých predikcí (pravděpodobností).

#### Vstupy
| Název | Typ | Popis | Povinný |
|-------|-----|-------|---------|
| `action` | `str` | Název akce z možností `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` a `delete`. | Ano |
| `name` | `str` | Název klasifikátoru, který se má použít. | Ano |
| `quantum_classifier` | `str` | Typ klasifikátoru, tj. `QuantumEnhancedEnsembleClassifier`. Výchozí hodnota je `QuantumEnhancedEnsembleClassifier`. | Ne |
| `X_train` | `array` nebo `list` nebo `str` | Trénovací data. Může jít o pole NumPy, seznam nebo řetězec odkazující na název souboru ve sdíleném datovém adresáři. | Ano |
| `y_train` | `array` nebo `list` nebo `str` | Cílové hodnoty pro trénování. Může jít o pole NumPy, seznam nebo řetězec odkazující na název souboru ve sdíleném datovém adresáři. | Ano |
| `X_test` | `array` nebo `list` nebo `str` | Testovací data. Může jít o pole NumPy, seznam nebo řetězec odkazující na název souboru ve sdíleném datovém adresáři. | Ano |
| `fit_params` | `dictionary` | Další parametry předávané metodě `fit` klasifikátoru. | Ne |
| `options["save"]` | `boolean` | Zda se má natrénovaný klasifikátor uložit do sdíleného datového adresáře. Výchozí hodnota je `True`. | Ne |
| `options["out"]` | `str` | Název výstupního JSON souboru pro uložení predikcí ve sdíleném datovém adresáři. Pokud není zadán, predikce jsou vráceny ve výsledku úlohy. | Ne |

#### Použití

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load function
singularity = catalog.load("multiverse/singularity")

#### Ověření
- `name`:
    - Pokud je `options["save"]` nastaveno na `True`:
        - Název musí být jedinečný, řetězec o délce nejvýše 64 znaků.
        - Smí obsahovat pouze alfanumerické znaky a podtržítka.
        - Musí začínat písmenem a nesmí končit podtržítkem.
        - Ve sdíleném datovém adresáři nesmí existovat žádný klasifikátor se stejným názvem.

- `options["out"]`:
    - Název souboru musí být jedinečný, řetězec o délce nejvýše 64 znaků.
    - Smí obsahovat pouze alfanumerické znaky a podtržítka.
    - Musí začínat písmenem a nesmí končit podtržítkem.
    - Musí mít příponu `.json`.
---
## Začínáme
Ověř svou totožnost pomocí [klíče API platformy IBM Quantum](http://quantum.cloud.ibm.com/) a vyber Qiskit Function takto:

In [3]:
# import the necessary modules for this example
import os
import tarfile
import numpy as np

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# generate the synthetic dataset
X, y = make_moons(n_samples=10000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# print the first 10 samples of the training dataset
print("Features:", X_train[:10, :])
print("Targets:", y_train[:10])

Features: [[-0.99958218  0.02890441]
 [ 0.03285169  0.24578719]
 [ 1.13127903 -0.49134546]
 [ 1.86951286  0.00608971]
 [ 0.20190413  0.97940529]
 [ 0.8831311   0.46912627]
 [-0.10819442  0.99412975]
 [-0.20005727  0.97978421]
 [-0.78775705  0.61598607]
 [ 1.82453236 -0.0658148 ]]
Targets: [0 1 1 1 0 0 0 0 0 1]


## Příklad
V tomto příkladu použiješ funkci „Singularity Machine Learning - Classification" ke klasifikaci datasetu složeného ze dvou vzájemně propletených měsícovitých půlkružnic. Dataset je syntetický, dvourozměrný a označený binárními štítky. Je navržen tak, aby byl náročný pro algoritmy jako shlukování na základě centroidů nebo lineární klasifikace.
![Moons dataset](../docs/images/guides/multiverse-computing-singularity/moon_shaped.avif)
V průběhu tohoto procesu se naučíš, jak vytvořit klasifikátor, přizpůsobit ho trénovacím datům, použít ho k predikci na testovacích datech a po dokončení ho smazat.
Než začneš, musíš nainstalovat [scikit-learn](https://scikit-learn.org/). Nainstaluj ho pomocí následujícího příkazu:

In [4]:
def make_tarfile(file_path, tar_file_name):
    with tarfile.open(tar_file_name, "w") as tar:
        tar.add(file_path, arcname=os.path.basename(file_path))


# save the training and test datasets on your local disk
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

# create tar files for the datasets
make_tarfile("X_train.npy", "X_train.npy.tar")
make_tarfile("y_train.npy", "y_train.npy.tar")
make_tarfile("X_test.npy", "X_test.npy.tar")
make_tarfile("y_test.npy", "y_test.npy.tar")

# upload the datasets to the shared data directory
catalog.file_upload("X_train.npy.tar", singularity)
catalog.file_upload("y_train.npy.tar", singularity)
catalog.file_upload("X_test.npy.tar", singularity)
catalog.file_upload("y_test.npy.tar", singularity)

# view/enlist the uploaded files in the shared data directory
print(catalog.files(singularity))

['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar']


Proveď následující kroky:

1) Vytvoř syntetický dataset pomocí funkce [make_moons](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) z knihovny [scikit-learn](https://scikit-learn.org/).
2) Nahraj vygenerovaný syntetický dataset do [sdíleného datového adresáře](https://qiskit.github.io/qiskit-serverless/getting_started/experimental/manage_data_directory.html).
3) Vytvoř kvantově rozšířený klasifikátor pomocí akce [create](#2-create).
4) Zobraz seznam svých klasifikátorů pomocí akce [list](#1-list).
5) Natrénuj klasifikátor na trénovacích datech pomocí akce [fit](#4-fit).
6) Použij natrénovaný klasifikátor k predikci na testovacích datech pomocí akce [predict](#5-predict).
7) Smaž klasifikátor pomocí akce [delete](#3-delete).
8) Po dokončení ukliď po sobě.
**Krok 1.** Importuj potřebné moduly a vygeneruj syntetický dataset, poté ho rozděl na trénovací a testovací datasety.

In [5]:
job = singularity.run(
    action="create",
    name="my_classifier",
    num_learners=10,
    learners_types=[
        "DecisionTreeClassifier",
        "KNeighborsClassifier",
    ],
    learners_proportions=[0.5, 0.5],
    learners_options=[{}, {}],
    regularization=0.01,
    weight_update_method="logarithmic",
    sample_scaling=True,
    optimizer_options={"simulator": True},
    voting="soft",
    prob_threshold=0.5,
)

print(job.result())

{'status': 'ok', 'message': 'Classifier created.', 'data': {}, 'metadata': {'resource_usage': {}}}


In [6]:
# list available classifiers using the list action
job = singularity.run(action="list")

print(job.result())

# you can also find your classifiers in the shared data directory with a *.pkl.tar extension
print(catalog.files(singularity))

{'status': 'ok', 'message': 'Classifiers listed.', 'data': {'classifiers': ['my_classifier']}, 'metadata': {'resource_usage': {}}}
['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar', 'my_classifier.pkl.tar']


**Krok 2.** Ulož označené trénovací a testovací datasety na lokální disk a poté je nahraj do [sdíleného datového adresáře](https://qiskit.github.io/qiskit-serverless/getting_started/experimental/manage_data_directory.html).

In [7]:
job = singularity.run(
    action="fit",
    name="my_classifier",
    X="X_train.npy",  # you do not need to specify the tar extension
    y="y_train.npy",  # you do not need to specify the tar extension
)

print(job.result())

{'status': 'ok', 'message': 'Classifier fitted.', 'data': {}, 'metadata': {'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 8.45469617843628}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 69.4949426651001}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 73.01881957054138}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 75.4787163734436}}}}


**Step 5.** Obtain predictions and probabilities from the quantum-enhanced classifier using the [predict](#5-predict) action.

In [8]:
job = singularity.run(
    action="predict",
    name="my_classifier",
    X="X_test.npy",  # you do not need to specify the tar extension
)

result = job.result()

print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results):", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results):", result["data"]["probabilities"][:5]
)

Action result status:  ok
Action result message:  Classifier predicted.
Predictions (first five results): [0, 1, 0, 0, 1]
Probabilities (first five results): [[1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 0.0], [0.0, 1.0]]


**Krok 3.** Vytvoř klasifikátor s kvantovým vylepšením pomocí akce [create](#2-create).

In [9]:
job = singularity.run(
    action="delete",
    name="my_classifier",
)

# or you can delete from the shared data directory
# catalog.file_delete("my_classifier.pkl.tar", singularity)

print(job.result())

{'status': 'ok', 'message': 'Classifier deleted.', 'data': {}, 'metadata': {'resource_usage': {}}}


**Step 7.** Clean up local and shared data directories.

In [ ]:
# delete the numpy files from your local disk
os.remove("X_train.npy")
os.remove("y_train.npy")
os.remove("X_test.npy")
os.remove("y_test.npy")

# delete the tar files from your local disk
os.remove("X_train.npy.tar")
os.remove("y_train.npy.tar")
os.remove("X_test.npy.tar")
os.remove("y_test.npy.tar")

# delete the tar files from the shared data
catalog.file_delete("X_train.npy.tar", singularity)
catalog.file_delete("y_train.npy.tar", singularity)
catalog.file_delete("X_test.npy.tar", singularity)
catalog.file_delete("y_test.npy.tar", singularity)

## Benchmarks

These benchmarks show that the classifier can achieve extremely high accuracies on challenging problems. They also show that increasing the number of learners in the ensemble (number of qubits) can lead to increased accuracy.

"Classical accuracy" refers to the accuracy obtained using corresponding classical state of the art which, in this case, is an AdaBoost classifier based on an ensemble of size 75. "Quantum accuracy", on the other hand, refers to the accuracy obtained using the "Singularity Machine Learning - Classification".

| Problem | Dataset Size | Ensemble Size | Number of Qubits | Classical Accuracy | Quantum Accuracy | Improvement |
|-------------|-------------|-------------|-------------|-------------|-------------|-------------|
| Grid stability | 5000 examples, 12 features | 55 | 55 |  76% | 91% | 15% |
| Grid stability | 5000 examples, 12 features | 65 | 65 |  76% | 92% | 16% |
| Grid stability | 5000 examples, 12 features | 75 | 75 |  76% | 94% | 18% |
| Grid stability | 5000 examples, 12 features | 85 | 85 |  76% | 94% | 18% |
| Grid stability | 5000 examples, 12 features | 100 | 100 |  76% | 95% | 19% |

----

As quantum hardware evolves and scales, the implications for our quantum classifier become increasingly significant. While the number of qubits does impose limitations on the size of the ensemble that can be utilized, it does not restrict the volume of data that can be processed. This powerful capability enables the classifier to efficiently handle datasets containing millions of data points and thousands of features. Importantly, the constraints related to ensemble size can be addressed through the implementation of a large-scale version of the classifier. By leveraging an iterative outer-loop approach, the ensemble can be dynamically expanded, enhancing flexibility and overall performance. However, it's worth noting that this feature has not yet been implemented in the current version of the classifier.

## Changelog

### 4 June 2025
- Upgraded `QuantumEnhancedEnsembleClassifier` with the following updates:
  - Added onsite/alpha regularization. You can specify `regularization_type` to be `onsite` or `alpha`
  - Added auto-regularization. You can set `regularization` to `auto` to use auto-regularization
  - Added `optimization_data` parameter to the `fit` method to choose optimization data for quantum optimization. You can use one of these options: `train`, `validation`, or `both`
  - Improved overall performance
- Added detailed status tracking for running jobs

### 20 May 2025
- Standardized error handling

### 18 March 2025
- Upgraded qiskit-serverless to 0.20.0 and base image to 0.20.1

### 14 February 2025
- Upgraded base image to 0.19.1

### 6 February 2025
- Upgraded qiskit-serverless to 0.19.0 and base image to 0.19.0

### 13 November 2024
- Release of Singularity Machine Learning - Classification

## Get support

For any questions, [reach out to Multiverse Computing](mailto:singularity@multiversecomputing.com).

Be sure to include the following information:

- The Qiskit Function Job ID (`job.job_id`)
- A detailed description of the issue
- Any relevant error messages or codes
- Steps to reproduce the issue

## Next steps

<Admonition type="tip" title="Recommendations">

- Request access to [Multiverse Computing's Singularity Machine Learning Classification function](https://quantum.cloud.ibm.com/functions?id=multiverse-singularity).
- Review [Leclerc, L., et al. (2023). Financial risk management on a neutral atom quantum processor. Physical Review Research, 5, 043117.](https://journals.aps.org/prresearch/pdf/10.1103/PhysRevResearch.5.043117)

</Admonition>